<a href="https://colab.research.google.com/github/alejocast2511/LENGUAJE-DE-PROGRAMACION-DCS/blob/main/ESCENARIOS_DE_INVERSION_DESCUBRIENDO_TU_PERFIL_INVERSOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
"""
Proyecto: Análisis de portafolios por Simulación Monte Carlo
Formato: Script preparado para ejecutarse en Google Colab (o local) y desplegar resultados con Streamlit.
Instrucciones rápidas:
1) En Colab: ejecutar las celdas en orden. Para ver la app Streamlit en Colab puedes usar `pyngrok` o descargar y ejecutar localmente `streamlit run Proyecto_MonteCarlo_Portafolios.py`.
2) Librerías: yfinance, pandas, numpy, matplotlib, streamlit, scipy, pyngrok (opcional para Colab).

Estructura del archivo:
- Config e instalaciones
- Definición de carteras (tickers)
- Funciones: descarga datos, cálculo retornos, simulación Monte Carlo, resumen
- Lógica Streamlit: cuestionario para perfil de inversor -> seleccionar cartera -> correr simulación -> mostrar resultados

El código está documentado y listo para copiar en una celda de Colab o como archivo .py
"""

# ==== (1) Instalación de dependencias (ejecutar en Colab) ====
# En Colab, descomenta y ejecuta:
# !pip install yfinance streamlit pyngrok scipy

# ==== (2) Importar librerías ====
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import streamlit as st
import io

# ==== (3) Definición de carteras (ejemplo) ====
# Cada cartera es una lista de tickers representativos (puedes cambiarlos)
CARTERAS = {
    'Conservadora': {
        'tickers': ['BND', 'IEF', 'VGSH', 'VTI'],  # bonos indexados y acciones de baja volatilidad
        'pesos': [0.60, 0.20, 0.10, 0.10]           # ejemplo: mayor exposición a renta fija
    },
    'Balanceada': {
        'tickers': ['AGG', 'TLT', 'VTI', 'VEA', 'VWO'],
        'pesos': [0.30, 0.10, 0.30, 0.20, 0.10]
    },
    'Arriesgada': {
        'tickers': ['VTI', 'QQQ', 'VOX', 'VWO', 'GDX'],
        'pesos': [0.40, 0.25, 0.10, 0.15, 0.10]
    }
}

# ==== (4) Funciones utilitarias ====

def descargar_datos(tickers, periodo='5y'):
    """Descarga precios ajustados históricos y devuelve DataFrame de precios.
    periodo: '1y','3y','5y','10y' o 'max'"""
    data = yf.download(tickers, period=periodo, interval='1d', progress=False)['Adj Close']
    # Si sólo un ticker, asegurar DataFrame
    if isinstance(data, pd.Series):
        data = data.to_frame()
    data = data.dropna(how='all')
    return data


def calcular_retornos_log(precios):
    """Retornos logarítmicos diarios"""
    return np.log(precios / precios.shift(1)).dropna()


def simulacion_montecarlo(retornos, pesos, dias_horizonte=252, n_simul=5000, pt_inicial=100000):
    """Simula n_simul escenarios de valor final del portafolio usando distribución normal multivar.
    - retornos: DataFrame de retornos diarios históricos
    - pesos: lista o array con pesos que suman 1
    Devuelve array de valores finales y estadísticos básicos."""
    mu = retornos.mean().values * 252  # retorno esperado anual
    sigma = retornos.cov().values * 252  # covarianza anualizada
    chol = np.linalg.cholesky(sigma)

    n_assets = len(mu)
    resultados_final = np.zeros(n_simul)
    trayectorias = np.zeros((n_simul, dias_horizonte))

    for i in range(n_simul):
        # simular incrementos diarios mediante proceso gaussiano multivar
        z = np.random.normal(size=(dias_horizonte, n_assets))
        shocks = z.dot(chol.T)  # shape (dias_horizonte, n_assets)
        # convertir a retornos diarios usando aproximación normal
        dt = 1/252
        series = np.exp((mu/252 - 0.5 * np.diag(sigma)/252) * 1 + shocks * np.sqrt(dt))
        # cada día, factor de crecimiento por activo
        precios_relativos = np.cumprod(series, axis=0)
        # valor diario del portafolio
        valores = (precios_relativos * pesos).sum(axis=1) * pt_inicial
        resultados_final[i] = valores[-1]
        trayectorias[i, :] = valores

    return {
        'final_values': resultados_final,
        'trayectorias': trayectorias,
        'mean_final': np.mean(resultados_final),
        'median_final': np.median(resultados_final),
        'std_final': np.std(resultados_final),
        'percentiles': np.percentile(resultados_final, [5,25,50,75,95])
    }

# ==== (5) Mapeo de cuestionario a perfil de inversión ====

def determinar_perfil(respuestas):
    """Respuestas: dict con claves:
    - horizonte (años)
    - aversion_perdida (1-5) donde 5 = muy averso
    - objetivo (crecimiento, ingresos, preservacion)
    - tolerancia_vol (baja-media-alta)
    Retorna: 'Conservadora'|'Balanceada'|'Arriesgada'"""
    horizonte = respuestas.get('horizonte', 5)
    aversion = respuestas.get('aversion_perdida', 3)
    objetivo = respuestas.get('objetivo', 'crecimiento')
    tolerancia = respuestas.get('tolerancia_vol', 'media')

    score = 0
    # Horizonte más largo -> preferencia por riesgo
    if horizonte >= 10:
        score += 2
    elif horizonte >=5:
        score += 1
    # aversion: alto -> menos riesgo
    score += (3 - aversion)  # aversion=5 -> -2 -> baja puntuación
    if objetivo == 'preservacion':
        score -= 2
    if tolerancia == 'baja':
        score -= 1
    elif tolerancia == 'alta':
        score += 1

    if score <= -1:
        return 'Conservadora'
    elif score <= 2:
        return 'Balanceada'
    else:
        return 'Arriesgada'

# ==== (6) Streamlit App ====

def run_app():
    st.title('Proyecto: Simulación Monte Carlo para Análisis de Portafolios')
    st.write('Esta app identifica el perfil del inversor mediante preguntas y simula 5,000 escenarios del portafolio recomendado.')

    st.sidebar.header('Cuestionario - Identificación de perfil')
    horizonte = st.sidebar.selectbox('Horizonte de inversión (años)', [1,2,3,5,7,10,15,20], index=3)
    aversion = st.sidebar.slider('¿Qué tanto te afecta una pérdida en tu inversión? (1: poco, 5: mucho)', 1, 5, 3)
    objetivo = st.sidebar.selectbox('Objetivo principal', ['crecimiento', 'ingresos', 'preservacion'], index=0)
    tolerancia = st.sidebar.selectbox('Tolerancia a la volatilidad', ['baja','media','alta'], index=1)
    monto = st.sidebar.number_input('Capital a invertir (USD)', min_value=1000, value=100000, step=1000)

    respuestas = {
        'horizonte': horizonte,
        'aversion_perdida': aversion,
        'objetivo': objetivo,
        'tolerancia_vol': tolerancia
    }

    perfil = determinar_perfil(respuestas)
    st.sidebar.markdown(f'**Perfil estimado:** {perfil}')

    periodo_hist = st.sidebar.selectbox('Periodo histórico para estimar retornos', ['1y','3y','5y','10y'], index=2)
    n_simul = st.sidebar.slider('Número de simulaciones', 1000, 20000, 5000, step=500)
    dias_horizonte = st.sidebar.slider('Horizonte de simulación (días)', 30, 2520, 252, step=30)

    if st.button('Ejecutar simulación'):
        cartera = CARTERAS[perfil]
        st.write('Cartera seleccionada:', perfil)
        st.write('Composición (tickers):', cartera['tickers'])

        with st.spinner('Descargando datos...'):
            precios = descargar_datos(cartera['tickers'], periodo=periodo_hist)
        st.write('Datos descargados. Periodos con datos disponibles:', precios.index.min().date(), 'a', precios.index.max().date())

        retornos = calcular_retornos_log(precios)
        pesos = np.array(cartera['pesos'])
        # normalizar pesos por si acaso
        pesos = pesos / pesos.sum()

        with st.spinner('Corriendo simulación Monte Carlo...'):
            resultado = simulacion_montecarlo(retornos, pesos, dias_horizonte, n_simul, monto)

        st.subheader('Resumen estadístico del valor final (USD)')
        st.write('- Valor medio final: ${:,.2f}'.format(resultado['mean_final']))
        st.write('- Mediana final: ${:,.2f}'.format(resultado['median_final']))
        p = resultado['percentiles']
        st.write('- Percentiles (5,25,50,75,95): ${:,.2f}, ${:,.2f}, ${:,.2f}, ${:,.2f}, ${:,.2f}'.format(*p))

        # Histograma final values
        fig, ax = plt.subplots()
        ax.hist(resultado['final_values'], bins=50)
        ax.set_title('Distribución del valor final del portafolio')
        ax.set_xlabel('Valor final (USD)')
        ax.set_ylabel('Frecuencia')
        st.pyplot(fig)

        # Mostrar trayectorias ejemplo
        fig2, ax2 = plt.subplots()
        for i in range(min(50, resultado['trayectorias'].shape[0])):
            ax2.plot(resultado['trayectorias'][i, :], alpha=0.4)
        ax2.set_title('Ejemplo de trayectorias de 50 simulaciones')
        ax2.set_xlabel('Días')
        ax2.set_ylabel('Valor del portafolio (USD)')
        st.pyplot(fig2)

        # Descargar resultados como CSV
        df_final = pd.DataFrame(resultado['final_values'], columns=['final_value'])
        csv = df_final.to_csv(index=False).encode('utf-8')
        st.download_button('Descargar valores finales (CSV)', data=csv, file_name='resultados_finales.csv', mime='text/csv')

# ==== (7) Punto de entrada para ejecución directa ====
if __name__ == '__main__':
    # Esto permite correr la app con: streamlit run Proyecto_MonteCarlo_Portafolios.py
    run_app()


2025-10-22 01:33:59.814 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 01:33:59.815 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 01:33:59.816 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 01:33:59.817 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 01:33:59.818 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 01:33:59.819 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 01:33:59.820 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 01:33:59.821 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar